# ConsentML in action: tracking an ML experiment and revoking consent

[ConsentML](https://github.com/KaranamLokesh/consentml) is a Python library that adds
**training-data lineage** and **consent-revocation reporting** to ML pipelines.

This notebook:

1. Trains two models on a synthetic customer dataset, with lineage tracked by the `@track` decorator.
2. Processes a customer's *right-to-be-forgotten* request with `revoke()`.
3. Shows the remediation recommendation change after the model is retrained without that customer.
4. Verifies the tamper-evident audit log — and demonstrates that tampering is detected.

**Setup:** run the next cell and upload the `consentml-0.1.0.dev0-py3-none-any.whl` wheel when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()  # upload consentml-0.1.0.dev0-py3-none-any.whl
%pip install -q consentml-0.1.0.dev0-py3-none-any.whl

import consentml
print("consentml", consentml.__version__, "installed")

## 1. A synthetic customer dataset

200 customers with spend, support and tenure features, and a churn label.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

DB = "lineage.db"
Path(DB).unlink(missing_ok=True)  # start fresh so the notebook is re-runnable

n = 200
customers = pd.DataFrame(
    {
        "email": [f"user{i:03d}@example.com" for i in range(n)],
        "monthly_spend": rng.gamma(2.0, 40.0, n).round(2),
        "support_tickets": rng.poisson(1.5, n),
        "tenure_months": rng.integers(1, 60, n),
    }
)
customers["churned"] = (
    (customers["support_tickets"] > 2) & (customers["tenure_months"] < 24)
).astype(int)
customers.head()

## 2. Train models with lineage tracking

One decorator on the training function is the whole integration. ConsentML records which
subjects' data went into the run, hashes the trained model, and appends to the audit log —
subject IDs are stored as SHA-256 hashes, never raw emails.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from consentml import track

FEATURES = ["monthly_spend", "support_tickets", "tenure_months"]


@track(
    data_source="warehouse://demo/customers",
    subject_id_col="email",
    model_name="churn_predictor",
    db_path=DB,
)
def train_churn_model(df):
    model = RandomForestClassifier(n_estimators=50, random_state=0)
    model.fit(df[FEATURES], df["churned"])
    return model


churn_model = train_churn_model(customers)
print("churn_predictor trained on", len(customers), "customers,",
      "accuracy:", round(churn_model.score(customers[FEATURES], customers["churned"]), 3))

In [ ]:
upsell_df = customers.sample(120, random_state=1).copy()
upsell_df["will_upsell"] = (upsell_df["monthly_spend"] > 80).astype(int)


@track(
    data_source="warehouse://demo/customers",
    subject_id_col="email",
    model_name="upsell_scorer",
    db_path=DB,
)
def train_upsell_model(df):
    model = LogisticRegression(max_iter=500)
    model.fit(df[FEATURES], df["will_upsell"])
    return model


upsell_model = train_upsell_model(upsell_df)
print("upsell_scorer trained on", len(upsell_df), "customers")

## 3. What did ConsentML record?

A local SQLite lineage store: one row per training run, one index row per subject.

In [ ]:
import sqlite3

conn = sqlite3.connect(DB)
runs = pd.read_sql(
    "SELECT model_name, n_subjects, substr(model_hash, 1, 12) AS model_hash, started_at "
    "FROM training_runs", conn)
n_index = pd.read_sql("SELECT count(*) AS subject_index_rows FROM subject_index", conn)
conn.close()

print(runs.to_string(index=False))
print()
print(n_index.to_string(index=False))

## 4. A customer revokes consent

We pick a customer who is in **both** training sets. `revoke()` reports every model trained
on their data, with a per-model recommendation — it never deletes or modifies anything;
the operator stays in control.

In [ ]:
import json

from consentml import revoke

revoked_email = upsell_df["email"].iloc[0]  # present in both models' training data
print("Customer revoking consent:", revoked_email)
print()

report = revoke(subject_id=revoked_email, db_path=DB)
print(json.dumps(report.to_dict(), indent=2))

Both models are flagged `retrain`: the customer's data is in the **latest** run of each.

## 5. Remediate, then check again

Retrain the churn model *without* the revoked customer, then re-run the check
(`dry_run=True` reports without recording a second revocation event).

In [ ]:
remaining = customers[customers["email"] != revoked_email]
churn_model_v2 = train_churn_model(remaining)
print("churn_predictor retrained on", len(remaining), "customers")
print()

report2 = revoke(subject_id=revoked_email, db_path=DB, dry_run=True)
for action in report2.recommended_actions:
    print(action)

The recommendation for `churn_predictor` flipped to `review` — the customer's data only
appears in a *superseded* run, so the operator just needs to confirm the old artifact is no
longer deployed. `upsell_scorer` still needs a retrain.

## 6. The tamper-evident audit log

Every event is hash-chained (like a mini certificate-transparency log). Let's verify the
chain, then tamper with the database and watch verification fail.

In [ ]:
import hashlib


def verify_chain(db_path):
    conn = sqlite3.connect(db_path)
    entries = conn.execute(
        "SELECT timestamp, event_type, payload, prev_hash, entry_hash "
        "FROM audit_log ORDER BY id"
    ).fetchall()
    conn.close()
    prev = "0" * 64
    for i, (ts, event, payload, prev_hash, entry_hash) in enumerate(entries, 1):
        recomputed = hashlib.sha256(
            (prev_hash + ts + event + payload).encode()
        ).hexdigest()
        if prev_hash != prev or entry_hash != recomputed:
            print(f"FAIL entry #{i} ({event}): chain broken or contents modified")
            return False
        prev = entry_hash
        print(f"ok   entry #{i}  {event}")
    print(f"\nAudit chain verified: {len(entries)} entries intact.")
    return True


assert verify_chain(DB)

In [ ]:
# An attacker (or a bug) edits history: rename a model in the first audit entry.
conn = sqlite3.connect(DB)
conn.execute(
    "UPDATE audit_log SET payload = replace(payload, 'churn_predictor', 'other_model') "
    "WHERE id = 1"
)
conn.commit()
conn.close()

assert not verify_chain(DB), "tampering should have been detected!"
print("\nTampering detected — the hash chain caught the edit.")

## 7. The same workflow from the CLI

In [ ]:
!consentml revoke --subject-id user007@example.com --db lineage.db --dry-run --json

## Wrap-up

- `@track` gave us lineage **by construction** — no pipeline rework.
- `revoke()` answered "*which models learned from this person?*" with an auditable report.
- The hash-chained audit log makes the compliance trail tamper-evident.

ConsentML is a lineage and reporting tool, not a deletion tool: it tells you which models a
user touched, and records what you decided to do about it. MIT-licensed, local-only, zero
cloud dependencies.